**Predicting Hospital Re-Admission using Logistic Regression in MIMIC-IV Data<br>
AI380H Self-Learning Tutorial<br>
Fall 2026<br>**


## Import MIMIC-IV Data from BigQuery



In [1]:
import numpy as np
import pandas as pd

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
from google.cloud import bigquery
client = bigquery.Client(project="YOUR_PROJECT_ID")

### Admissions Table

In [4]:
# For each hospital admission, also get the time of the patient's previous admit/discharge and next admit/discharge (if exists)
# Exclude observation admissions, keep inpatient admissions only
# Excludie admissions where the patient died in the hospital (hospital_expire_flag = 0)
query = """
  WITH inpatient_admissions AS (
    SELECT *
    FROM `physionet-data.mimiciv_3_1_hosp.admissions`
    WHERE admission_type NOT LIKE '%OBSERVATION%'
  )
  , timeline AS (
    SELECT
      subject_id,
      hadm_id,
      ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY admittime, hadm_id) AS admission_number,
      COUNT(*) OVER (PARTITION BY subject_id ORDER BY UNIX_SECONDS(TIMESTAMP(admittime))RANGE BETWEEN 31536000 PRECEDING AND 1 PRECEDING) AS count_admissions_past_year,
      LAG(admittime) OVER (PARTITION BY subject_id order by admittime, hadm_id) AS previous_admittime,
      LAG(dischtime) OVER (PARTITION BY subject_id order by admittime, hadm_id) AS previous_dischtime,
      admittime,
      dischtime,
      LEAD(admittime) OVER (PARTITION BY subject_id order by admittime, hadm_id) AS next_admittime,
      LEAD(dischtime) OVER (PARTITION BY subject_id order by admittime, hadm_id) AS next_dischtime,
      * EXCEPT (subject_id, hadm_id, admittime, dischtime)
    FROM inpatient_admissions
  )
  SELECT *
  FROM timeline
  WHERE hospital_expire_flag = 0
  ORDER BY subject_id, admittime, hadm_id
"""
admissions = client.query(query).to_dataframe()

In [5]:
admissions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300568 entries, 0 to 300567
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   subject_id                  300568 non-null  Int64         
 1   hadm_id                     300568 non-null  Int64         
 2   admission_number            300568 non-null  Int64         
 3   count_admissions_past_year  300568 non-null  Int64         
 4   previous_admittime          150577 non-null  datetime64[us]
 5   previous_dischtime          150577 non-null  datetime64[us]
 6   admittime                   300568 non-null  datetime64[us]
 7   dischtime                   300568 non-null  datetime64[us]
 8   next_admittime              155280 non-null  datetime64[us]
 9   next_dischtime              155280 non-null  datetime64[us]
 10  deathtime                   0 non-null       datetime64[us]
 11  admission_type              300568 non-

### Patients Table

In [6]:
query = """
  SELECT *
  FROM `physionet-data.mimiciv_3_1_hosp.patients`
"""
patients = client.query(query).to_dataframe()

In [7]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   subject_id         364627 non-null  Int64 
 1   gender             364627 non-null  object
 2   anchor_age         364627 non-null  Int64 
 3   anchor_year        364627 non-null  Int64 
 4   anchor_year_group  364627 non-null  object
 5   dod                38301 non-null   dbdate
dtypes: Int64(3), dbdate(1), object(2)
memory usage: 17.7+ MB


### Prescriptions

In [8]:
# Create a count of medications prescribed during the final 24h of admission as a measure of # meds at discharge
query = """
WITH eligible_prescriptions AS (
    SELECT p.*
    FROM `physionet-data.mimiciv_3_1_hosp.prescriptions` AS p
    INNER JOIN `physionet-data.mimiciv_3_1_hosp.pharmacy` AS ph
        ON p.pharmacy_id = ph.pharmacy_id
    INNER JOIN `physionet-data.mimiciv_3_1_hosp.admissions` AS a
        ON p.hadm_id = a.hadm_id
    WHERE TIMESTAMP(ph.entertime) <= TIMESTAMP(a.dischtime)
)
SELECT
    a.hadm_id,
    COUNT(DISTINCT p.pharmacy_id) AS prescription_orders_recorded,
    COUNT(DISTINCT IF(
        p.drug_type = 'MAIN'
        AND TIMESTAMP(p.starttime) < TIMESTAMP(a.dischtime)
        AND (p.stoptime IS NULL
            OR TIMESTAMP(p.stoptime) > GREATEST(
                TIMESTAMP(a.admittime),
                TIMESTAMP_SUB(
                    TIMESTAMP(a.dischtime), INTERVAL 24 HOUR
                )
            )
        ),
        NULLIF(LOWER(TRIM(p.drug)), ''), NULL
    )) AS medication_count_last_24h
FROM `physionet-data.mimiciv_3_1_hosp.admissions` AS a
LEFT JOIN eligible_prescriptions AS p
    ON a.hadm_id = p.hadm_id
GROUP BY a.hadm_id
"""
rx_count = client.query(query).to_dataframe()

In [9]:
rx_count.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 3 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   hadm_id                       546028 non-null  Int64
 1   prescription_orders_recorded  546028 non-null  Int64
 2   medication_count_last_24h     546028 non-null  Int64
dtypes: Int64(3)
memory usage: 14.1 MB


### Lab Events

In [10]:
# Get the latest available hemoglobin value before discharge (within 36h of discharge)
query = """
  SELECT
      a.hadm_id,
      le.valuenum AS last_hemoglobin_before_discharge,
      le.valueuom AS hemoglobin_unit,
      TIMESTAMP_DIFF(TIMESTAMP(a.dischtime), TIMESTAMP(le.charttime), SECOND) / 3600.0 AS hemoglobin_hours_before_discharge
  FROM `physionet-data.mimiciv_3_1_hosp.admissions` AS a
  INNER JOIN `physionet-data.mimiciv_3_1_hosp.labevents` AS le
      ON a.hadm_id = le.hadm_id
      AND a.subject_id = le.subject_id
  WHERE le.itemid = 51222
      AND le.valuenum IS NOT NULL
      AND le.valuenum > 0
      AND LOWER(TRIM(le.valueuom)) = 'g/dl'
      AND TIMESTAMP(le.charttime) >= TIMESTAMP(a.admittime)
      AND TIMESTAMP(le.charttime) <= TIMESTAMP(a.dischtime)
      AND TIMESTAMP(le.storetime) <= TIMESTAMP(a.dischtime)
      AND TIMESTAMP(le.charttime) >= TIMESTAMP_SUB(TIMESTAMP(a.dischtime), INTERVAL 36 HOUR)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY a.hadm_id ORDER BY le.charttime DESC, le.storetime DESC, le.labevent_id DESC) = 1
"""
hemoglobin = client.query(query).to_dataframe()

In [11]:
hemoglobin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336295 entries, 0 to 336294
Data columns (total 4 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   hadm_id                            336295 non-null  Int64  
 1   last_hemoglobin_before_discharge   336295 non-null  float64
 2   hemoglobin_unit                    336295 non-null  object 
 3   hemoglobin_hours_before_discharge  336295 non-null  float64
dtypes: Int64(1), float64(2), object(1)
memory usage: 10.6+ MB


### Service

In [12]:
# Gather data on the first recorded service for each hospitalization
query = """
  SELECT
      s.hadm_id,
      s.curr_service AS admission_service
  FROM `physionet-data.mimiciv_3_1_hosp.services` AS s
  INNER JOIN `physionet-data.mimiciv_3_1_hosp.admissions` AS a
      ON s.hadm_id = a.hadm_id
  WHERE s.transfertime <= a.dischtime
  QUALIFY ROW_NUMBER() OVER (PARTITION BY s.hadm_id ORDER BY s.transfertime, s.curr_service) = 1
"""
services = client.query(query).to_dataframe()

In [13]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545693 entries, 0 to 545692
Data columns (total 2 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   hadm_id            545693 non-null  Int64 
 1   admission_service  545693 non-null  object
dtypes: Int64(1), object(1)
memory usage: 8.8+ MB


### Procedures

In [14]:
query = """
  SELECT
      hadm_id,
      COUNT(*) AS count_billed_procedures
  FROM `physionet-data.mimiciv_3_1_hosp.procedures_icd`
  GROUP BY hadm_id
"""
procedure_count = client.query(query).to_dataframe()

In [15]:
procedure_count.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287504 entries, 0 to 287503
Data columns (total 2 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   hadm_id                  287504 non-null  Int64
 1   count_billed_procedures  287504 non-null  Int64
dtypes: Int64(2)
memory usage: 4.9 MB


## Feature Engineering

### Outcome Variable - Readmission within 30 days

In [16]:
# Calculate days from discharge until next admission
# 86400 seconds in a day
admissions['days_til_next_admit'] = (admissions['next_admittime'] - admissions['dischtime']).dt.total_seconds() / 86400

# Visualize distribution
admissions['days_til_next_admit'].describe()


,days_til_next_admit
count,155280.000000
mean,417.875752
std,700.836829
min,-29.731250
25%,21.224306
50%,103.705556
75%,496.296181
max,5298.825694


Some patients seem to have overlapping admissions (next admission starts before previous discharge). Let's count these and filter them out.

In [17]:
sum(admissions['days_til_next_admit'] < 0)

10

In [18]:
# Filter out overlapping admissions without filtering out patients with no next admission
admissions = admissions.loc[(admissions['days_til_next_admit'] > 0) | admissions['next_admittime'].isna()]

# Create a flag for next admission within 30 days
admissions['readmit_within_30_days'] = 1 * ((admissions['days_til_next_admit'] > 0) & (admissions['days_til_next_admit'] <= 30))

In [19]:
admissions['readmit_within_30_days'].value_counts()

,count
readmit_within_30_days,
0,253728
1,46585


### Number of previous admissions

In [20]:
# Total previous admissions
admissions['count_prior_admissions'] = admissions['admission_number'] - 1
admissions['count_prior_admissions'].describe()

,count_prior_admissions
count,300313.0
mean,1.889852
std,4.005765
min,0.0
25%,0.0
50%,1.0
75%,2.0
max,71.0


In [21]:
# Admissions in the past year
admissions['count_admissions_past_year'].describe()

,count_admissions_past_year
count,300313.0
mean,0.825189
std,1.777151
min,0.0
25%,0.0
50%,0.0
75%,1.0
max,42.0


In [22]:
# Admissions prior to the past year
admissions['prior_admissions_before_past_year'] = (admissions['count_prior_admissions'] - admissions['count_admissions_past_year'])
admissions['prior_admissions_before_past_year'].describe()

,prior_admissions_before_past_year
count,300313.0
mean,1.064663
std,3.086809
min,0.0
25%,0.0
50%,0.0
75%,1.0
max,59.0


### Length of admission

In [23]:
admissions['length_of_stay'] = (admissions['dischtime'] - admissions['admittime']).dt.total_seconds() / 86400
admissions['length_of_stay'].describe()

,length_of_stay
count,300313.000000
mean,5.897458
std,7.536021
min,-0.450694
25%,2.229167
50%,3.862500
75%,6.725694
max,321.579167


Count and remove stays with discharge dates prior to admit dates

In [24]:
sum(admissions['length_of_stay'] < 0)

9

In [25]:
admissions = admissions.loc[admissions['length_of_stay'] > 0]

### Admission Type

In [26]:
admissions['admission_type'].value_counts()

,count
admission_type,
EW EMER.,170625
URGENT,52424
SURGICAL SAME DAY ADMISSION,42745
DIRECT EMER.,21473
ELECTIVE,13035


### Admission Location

In [27]:
admissions['admission_location'].value_counts()

,count
admission_location,
EMERGENCY ROOM,132284
PHYSICIAN REFERRAL,103364
TRANSFER FROM HOSPITAL,34232
WALK-IN/SELF REFERRAL,9476
CLINIC REFERRAL,6481
INTERNAL TRANSFER TO OR FROM PSYCH,5441
PROCEDURE SITE,3693
TRANSFER FROM SKILLED NURSING FACILITY,2657
PACU,2385


### Discharge Location

In [28]:
admissions['discharge_location'].value_counts()

,count
discharge_location,
HOME,154890
HOME HEALTH CARE,74631
SKILLED NURSING FACILITY,40522
REHAB,10906
CHRONIC/LONG TERM ACUTE CARE,7183
HOSPICE,3614
AGAINST ADVICE,2423
PSYCH FACILITY,1887
ACUTE HOSPITAL,1791


In [29]:
sum(admissions['discharge_location'].isna())

498

In [30]:
admissions["discharge_location"] = admissions["discharge_location"].fillna("Unknown")

In [31]:
# Filter out admission location with discharge location of "died"
admissions = admissions.loc[admissions['discharge_location'] != 'DIED']

### Estimated Age at Each Admission

In [32]:
# Inner join patient data with admissions data
df = pd.merge(admissions, patients, on = 'subject_id', how = 'inner')

# Calcuate estimated age at each admission based on anchor age
df['age_at_admit'] = df['anchor_age'] + (df['admittime'].dt.year - df['anchor_year'])

### Gender

In [33]:
df['gender'].value_counts()

,count
gender,
F,159839
M,140279


### Number of prescriptions near discharge


In [34]:
df = pd.merge(df, rx_count, on = 'hadm_id', how = 'left', validate="one_to_one")
assert df["medication_count_last_24h"].notna().all()
df['medication_count_last_24h'].describe()

,medication_count_last_24h
count,300118.0
mean,15.855154
std,6.390019
min,0.0
25%,11.0
50%,15.0
75%,20.0
max,56.0


### Last hemoglobin before discharge

In [35]:
df = pd.merge(df, hemoglobin, on = 'hadm_id', how = 'left')
df["hemoglobin_missing"] = df['last_hemoglobin_before_discharge'].isna().astype(int)
df['last_hemoglobin_before_discharge'].describe()

,last_hemoglobin_before_discharge
count,212103.000000
mean,10.748707
std,1.978716
min,2.600000
25%,9.200000
50%,10.600000
75%,12.100000
max,21.500000


### Services

In [36]:
df = pd.merge(df, services, on = 'hadm_id', how = 'left')
df['admission_service'].value_counts()

,count
admission_service,
MED,109296
SURG,30467
CMED,28199
OBS,21950
ORTHO,18652
OMED,18388
NMED,13940
NSURG,11612
PSYCH,9114


### Procedure Count

In [37]:
df = pd.merge(df, procedure_count, on = 'hadm_id', how = 'left')
# No matching procedure records means zero recorded billed procedures
df["count_billed_procedures"] = (df["count_billed_procedures"].fillna(0).astype(int))
df['count_billed_procedures'].describe()

,count_billed_procedures
count,300118.000000
mean,2.012598
std,2.667966
min,0.000000
25%,0.000000
50%,1.000000
75%,3.000000
max,41.000000


## Final dataset

In [38]:
# Filter to just useful variables
df = df[[
    'subject_id',
    'hadm_id',
    'age_at_admit',
    'gender',
    'admission_type',
    'admission_location',
    'length_of_stay',
    'discharge_location',
    'count_admissions_past_year',
    'prior_admissions_before_past_year',
    'readmit_within_30_days',
    'medication_count_last_24h',
    'last_hemoglobin_before_discharge',
    'hemoglobin_missing',
    'admission_service',
    'count_billed_procedures'
  ]]

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300118 entries, 0 to 300117
Data columns (total 16 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   subject_id                         300118 non-null  Int64  
 1   hadm_id                            300118 non-null  Int64  
 2   age_at_admit                       300118 non-null  Int64  
 3   gender                             300118 non-null  object 
 4   admission_type                     300118 non-null  object 
 5   admission_location                 300117 non-null  object 
 6   length_of_stay                     300118 non-null  float64
 7   discharge_location                 300118 non-null  object 
 8   count_admissions_past_year         300118 non-null  Int64  
 9   prior_admissions_before_past_year  300118 non-null  Int64  
 10  readmit_within_30_days             300118 non-null  int64  
 11  medication_count_last_24h          3001

### Separate Predictors and Outcome, and Numeric and Categorical Variables

In [40]:
outcome = "readmit_within_30_days"

numeric_cols = [
    "age_at_admit",
    "length_of_stay",
    "count_admissions_past_year",
    "prior_admissions_before_past_year",
    "medication_count_last_24h",
    "last_hemoglobin_before_discharge",
    "count_billed_procedures"
]

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location",
    "discharge_location",
    'hemoglobin_missing',
    'admission_service'
]

predictors = numeric_cols + categorical_cols

X = df[predictors].copy()
y = df[outcome].astype(int)

### Create training and test data splits

In [41]:
# Split groups by patient, so different admissions for the same patient cannot appear in both sets.
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["subject_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Confirm that no patient appears in both sets.
train_patients = set(df.iloc[train_idx]["subject_id"])
test_patients = set(df.iloc[test_idx]["subject_id"])

assert train_patients.isdisjoint(test_patients)

print(f"Training admissions: {len(X_train):,}")
print(f"Test admissions: {len(X_test):,}")
print(f"Training readmission rate: {y_train.mean():.1%}")
print(f"Test readmission rate: {y_test.mean():.1%}")

Training admissions: 240,168
Test admissions: 59,950
Training readmission rate: 15.6%
Test readmission rate: 15.3%


### Preprocess data

In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def normalize_predictors(data):
    data = data.copy()

    # Convert None, NaN, and pd.NA to "Unknown"
    # Consistent strings also prevent mixed category types
    for col in categorical_cols:
        data[col] = (
            data[col]
            .astype("string")
            .fillna("Unknown")
            .astype(object)
        )

    # Convert nullable numeric columns to ordinary floats
    # Missing numerical values remain NaN for median imputation
    for col in numeric_cols:
        data[col] = data[col].astype(float)

    return data

X_train = normalize_predictors(X_train)
X_test = normalize_predictors(X_test)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown",
    )),
    ("encoder", OneHotEncoder(
        drop="if_binary",
        handle_unknown="ignore",
    )),
])

# Apply the appropriate processing to each column group
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

## Split training data into fitting and validation sets by patient

In [43]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

groups_train = df.loc[X_train.index, "subject_id"]

validation_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=123,
)

fit_idx, val_idx = next(
    validation_split.split(X_train, y_train, groups=groups_train)
)

X_fit = X_train.iloc[fit_idx].copy()
X_val = X_train.iloc[val_idx].copy()

y_fit = y_train.iloc[fit_idx].copy()
y_val = y_train.iloc[val_idx].copy()

## Compare and Select an Evaluation Threshold



In [44]:
# Fit a fresh preprocessing-and-model pipeline
validation_model = clone(model_pipeline)
validation_model.fit(X_fit, y_fit)
val_prob = validation_model.predict_proba(X_val)[:, 1]

# Compare thresholds
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

rows = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    val_pred = (val_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, val_pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "percent_flagged": 100 * val_pred.mean(),
        "readmissions_caught": tp,
        "false_alarms": fp,
        "readmissions_missed": fn,
    })

threshold_results = pd.DataFrame(rows)
display(threshold_results.round(3))

,threshold,precision,recall,f1,percent_flagged,readmissions_caught,false_alarms,readmissions_missed
0,0.10,0.198,0.846,0.321,63.414,5993,24272,1090
1,0.15,0.267,0.622,0.374,34.543,4408,12078,2675
2,0.20,0.338,0.448,0.385,19.685,3171,6224,3912
3,0.25,0.388,0.334,0.359,12.769,2367,3727,4716
4,0.30,0.431,0.260,0.324,8.949,1840,2431,5243
5,0.40,0.490,0.156,0.236,4.717,1103,1148,5980
6,0.50,0.550,0.083,0.144,2.227,585,478,6498


In [45]:
# Evaluate model on test set with the threshold with the highest f1
chosen_threshold = threshold_results.loc[threshold_results["f1"].idxmax(), "threshold"]

final_model = clone(model_pipeline)
final_model.fit(X_train, y_train)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [46]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
)

# Probability-based metrics are shared by both thresholds
print(f"ROC-AUC: {roc_auc_score(y_test, test_prob):.3f}")
print(
    f"Average precision: "
    f"{average_precision_score(y_test, test_prob):.3f}"
)

for label, threshold in [
    ("Default", 0.50),
    ("Validation-selected", chosen_threshold),
]:
    predictions = (test_prob >= threshold).astype(int)

    print(f"\n{label} threshold: {threshold:.2f}")
    print(f"Admissions flagged: {predictions.mean():.1%}")
    print(confusion_matrix(y_test, predictions))
    print(classification_report(
        y_test,
        predictions,
        zero_division=0,
    ))

ROC-AUC: 0.726
Average precision: 0.358

Default threshold: 0.50
Admissions flagged: 2.8%
[[50039   731]
 [ 8250   930]]
              precision    recall  f1-score   support

           0       0.86      0.99      0.92     50770
           1       0.56      0.10      0.17      9180

    accuracy                           0.85     59950
   macro avg       0.71      0.54      0.54     59950
weighted avg       0.81      0.85      0.80     59950


Validation-selected threshold: 0.20
Admissions flagged: 20.9%
[[42564  8206]
 [ 4884  4296]]
              precision    recall  f1-score   support

           0       0.90      0.84      0.87     50770
           1       0.34      0.47      0.40      9180

    accuracy                           0.78     59950
   macro avg       0.62      0.65      0.63     59950
weighted avg       0.81      0.78      0.79     59950



## Inspect coefficients

In [47]:
feature_names = (final_model.named_steps["preprocessor"].get_feature_names_out())
coefficients = final_model.named_steps["model"].coef_[0]

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

display(coef_table.sort_values("coefficient", ascending=False).round(3))

,feature,coefficient
33,categorical__discharge_location_PSYCH FACILITY,1.348
49,categorical__admission_service_OMED,1.045
20,categorical__admission_location_PROCEDURE SITE,0.398
18,categorical__admission_location_PACU,0.380
2,numeric__count_admissions_past_year,0.373
24,categorical__discharge_location_ACUTE HOSPITAL,0.371
15,categorical__admission_location_EMERGENCY ROOM,0.285
53,categorical__admission_service_SURG,0.248
57,categorical__admission_service_VSURG,0.238
36,categorical__discharge_location_Unknown,0.203
